# 05 - Inference, decoding strategies, evaluation and error analysis

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
%matplotlib inline
from src import config as C
from src.inference import EventGenerator
from src.visualization import plot_attention
models = [d.name for d in sorted(C.MODELS_DIR.iterdir()) if (d / 'best.pt').exists()]
print(models)
gens = {m: EventGenerator(C.MODELS_DIR / m, 'cpu') for m in models}

## Inference on new events (no reference available)
title + types → serialise → normalise → tokenise → ids → encoder → `<sos>` → predict → feed back → … → `<eos>`.

In [ ]:
events = [("Concert de jazz au jardin", ["concert", "musique"]), ("Marché de Noël artisanal", ["marché", "vente"]),
          ("Atelier cuisine pour les enfants", ["atelier", "jeune public"])]
for title, types in events:
    print('INPUT:', title, types)
    for m, g in gens.items():
        for method in ['greedy', 'beam']:
            print(f'  {m:<10} {method:<6}', g.generate(title, types, method).text)
    print()

## Greedy vs beam search
Greedy: y_t = argmax P(y_t | y_<t, X): fast but locally optimal. Beam (width 3) keeps the 3 partial
sentences with the highest cumulative log-probability Σ log P; hypotheses ending in `<eos>` are set aside, and
the final choice is normalised by length ((5+|y|)/6)^0.7 so short outputs are not favoured unfairly.

In [ ]:
test = pd.read_csv(C.SPLIT_FILES['test'])
for i in [0, 7, 21]:
    r = test.iloc[i]; print('INPUT    :', r.input_text); print('REFERENCE:', r.target_text)
    for m, g in gens.items():
        for method in ['greedy', 'beam']:
            print(f'{m:<10}{method:<7}:', g.generate_from_text(r.input_text, method).text)
    print()

## Test-set metrics (from `scripts/04_evaluate.py`)

In [ ]:
cmp = pd.read_csv(C.METRICS_DIR / 'comparison.csv')
cmp[['model','decoding','n_test','bleu','rouge1','rouge2','rougeL','perplexity','token_acc','repetition_rate','distinct2','title_coverage','avg_gen_len','n_params','train_time_min']].round(3)

In [ ]:
from IPython.display import Image
for f in ['learning_curves.png', 'metric_comparison.png']:
    p = C.FIGURES_DIR / f
    if p.exists(): display(Image(p))

**Metrics.** BLEU-4 (n-gram precision + brevity penalty) and ROUGE-1/2/L (overlap / longest common
subsequence, F1) compare with ONE reference, while many different descriptions are valid for the same
event: low absolute values are expected and they mostly rank systems. Perplexity = exp(mean token NLL) under
teacher forcing; token accuracy = share of reference tokens predicted at the right position given the true
prefix (it ignores fluency and the error propagation of free generation). Repetition rate, distinct-2 and
title coverage are added to diagnose loops, genericity and the use of input information.

## Attention weights

In [ ]:
if 'attention' in gens:
    for i in [0, 7]:
        r = gens['attention'].generate_from_text(test.input_text[i], 'greedy')
        plot_attention(r.attention, r.source_tokens, r.tokens + ['<eos>'], title=test.input_text[i][:70]); plt.show()

Attention is often sharp on title words that are copied, and diffuse on function words
(the decoder's language model handles them). Attention weights show *where the decoder reads*, not a
causal explanation of the prediction.

## Side-by-side table and error analysis

In [ ]:
pd.read_csv(C.EXAMPLES_DIR / 'side_by_side.csv').head(10)

In [ ]:
pd.read_csv(C.METRICS_DIR / 'error_rates.csv')

In [ ]:
from IPython.display import Markdown
Markdown((C.EXAMPLES_DIR / 'error_analysis.md').read_text()[:12000])